# 3.3 Memory Systems for Conversational Applications

## Playground Notebook

LLMs are **fundamentally stateless** — every API call is isolated. Memory systems bridge this gap by storing, managing, and re-injecting conversation history into each prompt.

| Topic | What You'll Learn |
|-------|-------------------|
| **Why Memory Matters** | The statelessness problem and how memory solves it |
| **Buffer Memory** | Store everything — simple but overflows |
| **Window Memory** | Keep last K turns — fixed size, sharp cutoff |
| **Summary Memory** | LLM-compressed history — scales to long chats |
| **Summary Buffer Memory** | Hybrid: verbatim recent + summarized old — best default |
| **Conversational RAG** | Memory + retrieval + question reformulation |
| **Token Budget Management** | Controlling memory within context window limits |

### Memory Type Comparison

| Memory Type | Token Growth | Info Loss | LLM Overhead | Best For |
|-------------|-------------|-----------|-------------|----------|
| **Buffer** | Unbounded | None | None | Short chats, prototyping |
| **Window (K)** | Fixed | Sharp cutoff | None | Medium chats |
| **Token Buffer** | Fixed | Sharp cutoff | None | Variable-length messages |
| **Summary** | Slow growth | Moderate | 1 LLM call/turn | Long conversations |
| **Summary Buffer** | Very slow | Minimal | Occasional | **General purpose — best default** |
| **Entity** | Per-entity | Some | 2 LLM calls/turn | Entity-centric chats |
| **VectorStore** | Constant retrieval | Retrieval-dep. | Embedding/turn | Very long, topic-revisit |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. The Statelessness Problem

Every LLM API call is **completely isolated**. Without memory, the model forgets everything the moment you send a follow-up.

### The Memory Lifecycle

```
User Input
    ↓
Memory.load() → Conversation History
    ↓
Prompt Template
    ├── System Message (static instructions)
    ├── Conversation History (from memory)  ← MessagesPlaceholder
    └── Current User Message
    ↓
Chat Model → AI Response
    ↓
Memory.save(user_message, ai_response)
    ↓
Response to User
```

### Experiment 1A: Without Memory — The Problem

In [3]:
# Without memory — each call is isolated

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 1 sentence."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Two separate calls — no connection between them
print("Turn 1:")
print(f"  User: My name is Alex.")
print(f"  AI:   {chain.invoke({'question': 'My name is Alex. Remember it.'})}")

print("\nTurn 2:")
print(f"  User: What is my name?")
print(f"  AI:   {chain.invoke({'question': 'What is my name?'})}")

print("\nThe model has NO idea what your name is — each call is isolated!")

Turn 1:
  User: My name is Alex.
  AI:   I've taken note, Alex, and I'll remember your name for our conversation.

Turn 2:
  User: What is my name?
  AI:   I don't have any information about your name.

The model has NO idea what your name is — each call is isolated!


### Experiment 1B: With Memory — The Solution

In [4]:
# With memory — conversation history is passed each time

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 1 sentence."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

memory_chain = memory_prompt | llm | StrOutputParser()

# Manual memory — a list we manage ourselves
history = []

def chat(user_input):
    """Send message, get response, save to history."""
    response = memory_chain.invoke({
        "chat_history": history,
        "question": user_input
    })
    # Save the exchange
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Now it remembers!
print("Turn 1:")
print(f"  User: My name is Alex.")
print(f"  AI:   {chat('My name is Alex. Remember it.')}")

print("\nTurn 2:")
print(f"  User: What is my name?")
print(f"  AI:   {chat('What is my name?')}")

print(f"\nHistory size: {len(history)} messages")
print("Memory solved the problem by passing history with every call!")

Turn 1:
  User: My name is Alex.
  AI:   I've taken note that your name is Alex.

Turn 2:
  User: What is my name?
  AI:   Your name is Alex.

History size: 4 messages
Memory solved the problem by passing history with every call!


---

## 2. Memory Type 1: Buffer Memory — Store Everything

The simplest memory — stores every message exactly as it occurred. Perfect recall, but **grows unbounded**.

| Property | Value |
|----------|-------|
| Storage | Complete message list |
| Growth rate | Linear — grows every turn |
| Information loss | Zero — perfect recall |
| Computational overhead | None — just list append |
| Context window risk | **High — will eventually overflow** |

### The Overflow Problem

With an 8K token context window and ~200 tokens per turn:
- After 30 turns: ~6,000 tokens
- After 40 turns: ~8,000 tokens — **context window full!**
- After 41 turns: prompt fails or history must be truncated

### Experiment 2A: Buffer Memory in Action

In [5]:
# Buffer Memory — stores everything, perfect recall

class BufferMemory:
    """Stores all messages. Simple but grows unbounded."""

    def __init__(self):
        self.messages = []

    def load(self):
        """Return all stored messages."""
        return self.messages.copy()

    def save(self, human_msg, ai_msg):
        """Store the exchange."""
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

    def clear(self):
        self.messages = []

    @property
    def size(self):
        return len(self.messages)


# Wire it up with a chain
buffer_mem = BufferMemory()

convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly AI tutor. Keep answers to 1-2 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

convo_chain = convo_prompt | llm | StrOutputParser()

def chat_buffer(user_input):
    response = convo_chain.invoke({
        "history": buffer_mem.load(),
        "input": user_input
    })
    buffer_mem.save(user_input, response)
    return response


# Multi-turn conversation
turns = [
    "My name is Alex and I'm learning about memory in LangChain.",
    "What is the simplest type of memory?",
    "What's the main problem with it?",
    "What's my name and what am I learning?",  # Tests recall
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i} (history: {buffer_mem.size} msgs):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_buffer(msg)}")

print(f"\nFinal history: {buffer_mem.size} messages (all stored!)")
print("Buffer memory gives perfect recall but will overflow on long chats.")


Turn 1 (history: 0 msgs):
  User: My name is Alex and I'm learning about memory in LangChain.
  AI:   Hello Alex! LangChain is a popular open-source library for building conversational AI models, and memory is a crucial aspect of its architecture. How can I help you with understanding memory in LangChain?

Turn 2 (history: 2 msgs):
  User: What is the simplest type of memory?
  AI:   In LangChain, the simplest type of memory is a `SimpleMemory`, which is a basic, thread-safe, and immutable memory module that stores and retrieves data using a simple key-value pair interface.

Turn 3 (history: 4 msgs):
  User: What's the main problem with it?
  AI:   The main problem with `SimpleMemory` is that it doesn't allow for efficient retrieval of data by key, as it has to search the entire memory to find the value associated with a given key.

Turn 4 (history: 6 msgs):
  User: What's my name and what am I learning?
  AI:   Your name is Alex, and you're learning about memory in LangChain.

Final 

---

## 3. Memory Type 2: Window Memory — Last K Turns

Keeps only the **last K conversation exchanges**, creating a sliding window. Fixed size, predictable token usage.

| Property | Value |
|----------|-------|
| Storage | Full list, returns last K pairs |
| Growth rate | Fixed — always K × 2 messages |
| Information loss | Moderate — older context is **lost** |
| Context window risk | Low — predictable budget |

### Choosing K

| K Value | ~Tokens | Use Case |
|---------|---------|----------|
| K = 3 | ~600 | Quick Q&A bots |
| K = 5 | ~1,000 | Standard chatbots |
| K = 10 | ~2,000 | Detailed conversations |
| K = 15 | ~3,000 | Complex support |

**Rule of thumb:** Set K so memory uses no more than 25-30% of your context window.

### The Cliff Problem

Window memory has a **sharp cutoff** — information doesn't gradually fade, it disappears entirely. If the user said their name in Turn 1 and K=3, by Turn 5 it's completely gone.

### Experiment 3A: Window Memory — Sliding Window

In [6]:
# Window Memory — keeps only the last K exchanges

class WindowMemory:
    """Stores all messages but returns only the last K exchanges."""

    def __init__(self, k=3):
        self.messages = []
        self.k = k

    def load(self):
        """Return last K exchanges (K*2 messages)."""
        return self.messages[-(self.k * 2):]

    def save(self, human_msg, ai_msg):
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

    @property
    def total_stored(self):
        return len(self.messages)

    @property
    def window_size(self):
        return len(self.load())


window_mem = WindowMemory(k=2)  # Only keep last 2 exchanges

def chat_window(user_input):
    response = convo_chain.invoke({
        "history": window_mem.load(),
        "input": user_input
    })
    window_mem.save(user_input, response)
    return response


# Test the cliff problem
turns = [
    "My favorite color is blue. Remember that!",   # Turn 1 — will fall off
    "What is 2 + 2?",                               # Turn 2 — will fall off
    "What is the capital of France?",                # Turn 3
    "What is my favorite color?",                    # Turn 4 — Turn 1 is gone!
]

for i, msg in enumerate(turns, 1):
    window = window_mem.load()
    print(f"\nTurn {i} (stored: {window_mem.total_stored}, window sees: {window_mem.window_size}):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_window(msg)}")

print(f"\nTotal stored: {window_mem.total_stored} messages")
print(f"Window sees:  {window_mem.window_size} messages (last {window_mem.k} exchanges)")
print("\nThe 'cliff problem': early context doesn't fade — it vanishes entirely!")


Turn 1 (stored: 0, window sees: 0):
  User: My favorite color is blue. Remember that!
  AI:   I've taken note that your favorite color is blue! I'll keep that in mind for our conversation.

Turn 2 (stored: 2, window sees: 2):
  User: What is 2 + 2?
  AI:   The answer is 4, a simple math fact!

Turn 3 (stored: 4, window sees: 4):
  User: What is the capital of France?
  AI:   The capital of France is Paris, a beautiful and famous city.

Turn 4 (stored: 6, window sees: 4):
  User: What is my favorite color?
  AI:   I don't have any information about your favorite color, as I'm just a conversational AI and don't have the ability to read minds or keep track of personal preferences.

Total stored: 8 messages
Window sees:  4 messages (last 2 exchanges)

The 'cliff problem': early context doesn't fade — it vanishes entirely!


---

## 4. Memory Type 3: Token Buffer Memory — Precise Token Control

Like window memory but uses a **token count** instead of turn count. More precise for variable-length messages.

| Property | Value |
|----------|-------|
| Storage | Token-counted message buffer |
| Growth rate | Fixed — bounded by token limit |
| Information loss | Moderate — same cliff problem |
| Context window risk | Very low — exact token budget |

### Experiment 4A: Token Buffer Memory

In [7]:
# Token Buffer Memory — keeps messages that fit within a token budget
# We approximate tokens as words/0.75 (rough estimate for English text)

class TokenBufferMemory:
    """Keeps the most recent messages within a token budget."""

    def __init__(self, max_tokens=200):
        self.messages = []
        self.max_tokens = max_tokens

    def _estimate_tokens(self, text):
        """Rough token estimate: ~1 token per 4 characters."""
        return len(text) // 4

    def load(self):
        """Return most recent messages that fit within token budget."""
        result = []
        token_count = 0
        # Walk backwards from most recent
        for msg in reversed(self.messages):
            tokens = self._estimate_tokens(msg.content)
            if token_count + tokens > self.max_tokens:
                break
            result.insert(0, msg)
            token_count += tokens
        return result

    def save(self, human_msg, ai_msg):
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

    def get_token_usage(self):
        loaded = self.load()
        return sum(self._estimate_tokens(m.content) for m in loaded)


token_mem = TokenBufferMemory(max_tokens=150)

def chat_token(user_input):
    response = convo_chain.invoke({
        "history": token_mem.load(),
        "input": user_input
    })
    token_mem.save(user_input, response)
    return response


turns = [
    "My name is Alex and I work at a tech company in Chennai.",
    "I'm building a chatbot for customer support.",
    "We use Python and LangChain for the backend.",
    "What do you know about me?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i} (~{token_mem.get_token_usage()} tokens in memory):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_token(msg)}")

print(f"\nToken budget: {token_mem.max_tokens}")
print(f"Tokens used:  ~{token_mem.get_token_usage()}")
print(f"Messages in window: {len(token_mem.load())} of {len(token_mem.messages)} total")


Turn 1 (~0 tokens in memory):
  User: My name is Alex and I work at a tech company in Chennai.
  AI:   It's nice to meet you, Alex! Working in a tech company in Chennai must be exciting, with the city being a hub for innovation and technology in India.

Turn 2 (~51 tokens in memory):
  User: I'm building a chatbot for customer support.
  AI:   That's a great project, Alex! Building a chatbot to provide 24/7 customer support can help improve user experience and increase efficiency for your tech company.

Turn 3 (~102 tokens in memory):
  User: We use Python and LangChain for the backend.
  AI:   LangChain is a powerful library for building conversational AI models, and combining it with Python as your backend language is a great choice for developing a robust chatbot.

Turn 4 (~142 tokens in memory):
  User: What do you know about me?
  AI:   I know that you're building a chatbot for customer support using Python and LangChain, but I don't have any personal information about you beyond

---

## 5. Memory Type 4: Summary Memory — LLM-Compressed History

Uses an LLM to maintain a **running summary** instead of storing raw messages. Scales to very long conversations.

| Property | Value |
|----------|-------|
| Storage | Running text summary |
| Growth rate | Slow — summary grows logarithmically |
| Information loss | Some — exact wording and nuance lost |
| Computational overhead | **High — 1 LLM call per turn** |
| Context window risk | Very low |

### The Cost Tradeoff

Every turn requires an **extra LLM call** to update the summary. For 20 turns, that's 20 extra calls.

**Optimization:** Use a smaller, cheaper model for summaries while using a powerful model for conversation.

### Experiment 5A: Summary Memory

In [8]:
# Summary Memory — LLM maintains a running summary

class SummaryMemory:
    """Uses LLM to maintain a running conversation summary."""

    def __init__(self, llm):
        self.summary = ""
        self.llm = llm
        self.turn_count = 0
        self._summarize_chain = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Progressively summarize the conversation. "
                 "Add to the existing summary with new information. "
                 "Keep it concise — 2-3 sentences max."),
                ("human",
                 "Current summary: {summary}\n\n"
                 "New exchange:\n"
                 "Human: {human_msg}\n"
                 "AI: {ai_msg}\n\n"
                 "Updated summary:")
            ])
            | llm | StrOutputParser()
        )

    def load(self):
        """Return summary as a system-level context."""
        return self.summary

    def save(self, human_msg, ai_msg):
        """Update the running summary with the new exchange."""
        self.summary = self._summarize_chain.invoke({
            "summary": self.summary or "(No prior conversation)",
            "human_msg": human_msg,
            "ai_msg": ai_msg
        })
        self.turn_count += 1


summary_mem = SummaryMemory(llm)

# Prompt that uses summary as context
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly AI assistant. Keep answers to 1-2 sentences.\n"
     "Conversation so far: {summary}"),
    ("human", "{input}")
])

summary_chain = summary_prompt | llm | StrOutputParser()

def chat_summary(user_input):
    response = summary_chain.invoke({
        "summary": summary_mem.load() or "(New conversation)",
        "input": user_input
    })
    summary_mem.save(user_input, response)
    return response


turns = [
    "My name is Alex and I'm a data scientist.",
    "I'm building a chatbot for my company.",
    "We need it to handle 1000+ customer queries per day.",
    "The main topic is product returns and refunds.",
    "What do you know about me and my project?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_summary(msg)}")
    print(f"  Summary: {summary_mem.load()[:100]}...")

print(f"\n{'=' * 50}")
print("FINAL SUMMARY:")
print(f"{'=' * 50}")
print(summary_mem.load())
print(f"\nTurns: {summary_mem.turn_count}")
print("Summary grows slowly — scales to very long conversations!")


Turn 1:
  User: My name is Alex and I'm a data scientist.
  AI:   Nice to meet you, Alex! As a data scientist, I'm sure you're always looking for ways to improve your skills and stay up-to-date with the latest developments in the field.
  Summary: Alex, a data scientist, has begun a conversation with the AI, who has expressed interest in Alex's p...

Turn 2:
  User: I'm building a chatbot for my company.
  AI:   That's exciting! What specific features or functionalities are you looking to implement in your chatbot to enhance customer engagement and experience for your company?
  Summary: Current summary: Alex, a data scientist, has begun a conversation with the AI, who has expressed int...

Turn 3:
  User: We need it to handle 1000+ customer queries per day.
  AI:   Handling a high volume of customer queries daily requires a robust chatbot architecture, including features like natural language processing (NLP), advanced routing logic, and scalable backend infrastructure, to ensure eff

---

## 6. Memory Type 5: Summary Buffer Memory — The Best Default

The **hybrid approach** — combines verbatim recent messages with a summary of older messages. Widely considered the best general-purpose memory.

| Property | Value |
|----------|-------|
| Storage | Summary + recent message buffer |
| Growth rate | Very slow |
| Information loss | Minimal — exact recent + summarized old |
| Computational overhead | Medium — occasional LLM calls |

### Why This is the Best Default

| vs. Buffer | Handles long conversations without overflow |
|:-----------|:---|
| **vs. Window** | Doesn't lose old context — it's summarized, not deleted |
| **vs. Pure Summary** | Recent messages are exact (important for follow-ups) |
| **vs. Token Buffer** | Old context preserved in summary form |

The human memory pattern: you remember recent events in detail and older ones as general impressions.

### Experiment 6A: Summary Buffer Memory

In [9]:
# Summary Buffer Memory — verbatim recent + summarized old

class SummaryBufferMemory:
    """Hybrid: recent messages verbatim + older messages summarized."""

    def __init__(self, llm, max_recent=4):  # max_recent = number of messages (not turns)
        self.messages = []
        self.summary = ""
        self.max_recent = max_recent  # Keep last N messages verbatim
        self.llm = llm
        self._summarize_chain = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Progressively summarize the conversation. "
                 "Merge new messages into the existing summary. "
                 "2-3 sentences max."),
                ("human",
                 "Current summary: {summary}\n\n"
                 "New messages to incorporate:\n{new_messages}\n\n"
                 "Updated summary:")
            ])
            | llm | StrOutputParser()
        )

    def load(self):
        """Return summary + recent messages."""
        return {
            "summary": self.summary or "(No prior conversation)",
            "recent": self.messages[-self.max_recent:]
        }

    def save(self, human_msg, ai_msg):
        """Save exchange and summarize overflow."""
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

        # When messages exceed the buffer, summarize the overflow
        if len(self.messages) > self.max_recent:
            overflow = self.messages[:-(self.max_recent)]
            # Only summarize the oldest 2 messages that just aged out
            new_msgs_text = "\n".join(
                f"{m.type}: {m.content}" for m in overflow[-2:]
            )
            self.summary = self._summarize_chain.invoke({
                "summary": self.summary or "(none)",
                "new_messages": new_msgs_text
            })


sb_mem = SummaryBufferMemory(llm, max_recent=4)  # Keep last 2 exchanges verbatim

# Prompt with both summary and recent messages
sb_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly AI assistant. Keep answers to 1-2 sentences.\n"
     "Conversation so far: {summary}"),
    MessagesPlaceholder(variable_name="recent"),
    ("human", "{input}")
])

sb_chain = sb_prompt | llm | StrOutputParser()

def chat_summary_buffer(user_input):
    mem = sb_mem.load()
    response = sb_chain.invoke({
        "summary": mem["summary"],
        "recent": mem["recent"],
        "input": user_input
    })
    sb_mem.save(user_input, response)
    return response


turns = [
    "My name is Alex. I'm a Python developer.",
    "I'm building a customer support chatbot.",
    "We handle about 500 tickets per day.",
    "Most questions are about billing and refunds.",
    "We use LangChain with Ollama for the LLM.",
    "What do you know about me and my project?",  # Tests both summary + recent
]

for i, msg in enumerate(turns, 1):
    mem = sb_mem.load()
    print(f"\nTurn {i} (recent: {len(mem['recent'])} msgs, has summary: {bool(sb_mem.summary)}):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_summary_buffer(msg)}")

print(f"\n{'=' * 50}")
print("MEMORY STATE:")
print(f"{'=' * 50}")
print(f"Summary: {sb_mem.summary}")
print(f"Recent messages: {len(sb_mem.load()['recent'])}")
print(f"Total stored: {len(sb_mem.messages)}")
print("\nOld context is summarized, recent context is verbatim — best of both worlds!")


Turn 1 (recent: 0 msgs, has summary: False):
  User: My name is Alex. I'm a Python developer.
  AI:   Nice to meet you, Alex! As a Python developer, you must be familiar with the popular frameworks like Django, Flask, and Pyramid.

Turn 2 (recent: 2 msgs, has summary: False):
  User: I'm building a customer support chatbot.
  AI:   That's a great project, Alex! You'll likely be working with natural language processing (NLP) and machine learning libraries like NLTK, spaCy, or scikit-learn to create a conversational interface for your chatbot.

Turn 3 (recent: 4 msgs, has summary: False):
  User: We handle about 500 tickets per day.
  AI:   That's a significant volume, Alex! Handling 500 tickets per day will require a robust and efficient system to ensure timely responses and resolution of customer inquiries.

Turn 4 (recent: 4 msgs, has summary: True):
  User: Most questions are about billing and refunds.
  AI:   Billing and refunds can be complex topics, Alex, so it's likely that your

---

## 7. Memory Type 6: Entity Memory (Conceptual)

Tracks specific **entities** (people, places, products) mentioned across the conversation, instead of linear history.

| Property | Value |
|----------|-------|
| Storage | Entity-keyed knowledge base |
| Growth rate | Per-entity, not per-turn |
| Computational overhead | High — entity extraction per turn |

```
After a conversation about project planning:

Entities:
  "Project Alpha": "A mobile app due in March. Team of 5. Budget $50K."
  "Sarah":         "Project manager. Based in Chennai. Prefers weekly standups."
  "React Native":  "The framework for Project Alpha. Version 0.72."
```

### Experiment 7A: Simple Entity Tracker

In [10]:
# Simple entity memory — LLM extracts and updates entity descriptions

class SimpleEntityMemory:
    """Tracks entities mentioned in conversation."""

    def __init__(self, llm):
        self.entities = {}  # name → description
        self.llm = llm
        self._extract_chain = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Extract entity names and descriptions from this exchange. "
                 "Entities are people, projects, tools, or organizations.\n"
                 "Respond ONLY with valid JSON. No markdown.\n"
                 'Format: {{"entity_name": "description"}}\n'
                 "If no entities found, respond with {{}}"),
                ("human",
                 "Human: {human_msg}\nAI: {ai_msg}")
            ])
            | llm | StrOutputParser()
        )

    def load(self):
        if not self.entities:
            return "No entities tracked yet."
        return "\n".join(f"- {name}: {desc}" for name, desc in self.entities.items())

    def save(self, human_msg, ai_msg):
        import json
        try:
            raw = self._extract_chain.invoke({
                "human_msg": human_msg,
                "ai_msg": ai_msg
            })
            # Try to parse JSON from the response
            raw = raw.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1].strip().removeprefix("json").strip()
            new_entities = json.loads(raw)
            if isinstance(new_entities, dict):
                self.entities.update(new_entities)
        except (json.JSONDecodeError, Exception):
            pass  # Skip if parsing fails


entity_mem = SimpleEntityMemory(llm)

entity_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful project assistant. Keep answers to 1-2 sentences.\n"
     "Known entities:\n{entities}"),
    ("human", "{input}")
])

entity_chain = entity_prompt | llm | StrOutputParser()

def chat_entity(user_input):
    response = entity_chain.invoke({
        "entities": entity_mem.load(),
        "input": user_input
    })
    entity_mem.save(user_input, response)
    return response


turns = [
    "I'm Alex and I work at TechCorp on Project Alpha.",
    "Sarah is the project manager. She's based in Chennai.",
    "We're using React Native and the deadline is March.",
    "What do you know about Sarah and Project Alpha?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_entity(msg)}")

print(f"\n{'=' * 50}")
print("TRACKED ENTITIES:")
print(f"{'=' * 50}")
print(entity_mem.load())


Turn 1:
  User: I'm Alex and I work at TechCorp on Project Alpha.
  AI:   Hello Alex, I'm happy to help with Project Alpha. What can I assist you with today, from requirements gathering to task organization?

Turn 2:
  User: Sarah is the project manager. She's based in Chennai.
  AI:   Sarah is currently coordinating Project Alpha from TechCorp's Chennai office.

Turn 3:
  User: We're using React Native and the deadline is March.
  AI:   Project Alpha, a React Native project, has a tight deadline of March, which requires careful planning and execution to ensure timely completion. As a project assistant, I'll be happy to help with task management and coordination to meet the deadline.

Turn 4:
  User: What do you know about Sarah and Project Alpha?
  AI:   Sarah is the project manager overseeing Project Alpha, a React Native project with a tight deadline of March, and is likely responsible for ensuring the project stays on track and meets its goals.

TRACKED ENTITIES:
- Alex: person
- 

---

## 8. Conversational RAG — Memory + Retrieval

The most important production pattern: combine memory with retrieval AND **question reformulation**.

> **Note:** This is a simplified demo using keyword-based retrieval to illustrate how **memory integrates with RAG**. Full RAG with vector databases (FAISS, Chroma) and embeddings will be covered in the next unit.

### Why Reformulation is Mandatory

Turn 1: "What is LangChain?"      → Clear query, retrieval works
Turn 2: "Who created it?"          → "it" = ??? Retriever finds nothing!
Reformulated: "Who created LangChain?"  → Now retrieval works!



### Architecture

User Question
↓
Memory.load() → History
↓
Reformulation (history + question → standalone query)
↓
┌→ Retriever (standalone query) → Documents
└→ Passthrough → Standalone question
↓
RAG Prompt (system + docs + history + question)
↓
Chat Model → AI Response
↓
Memory.save()



### Token Budget Allocation for Conversational RAG

| Component | Typical Allocation |
|-----------|-------------------|
| System prompt | 300–500 tokens |
| Conversation history | 1,000–2,000 tokens |
| Retrieved documents | 2,000–3,000 tokens |
| Current question | 50–200 tokens |
| Response reserve | 1,000–2,000 tokens |

### Experiment 8A: Conversational RAG with Reformulation

In [11]:
# Conversational RAG with question reformulation
# NOTE: This uses simple keyword retrieval to demonstrate the MEMORY concept.
# Full RAG with vector databases will be covered in the next unit.

# Simulated knowledge base
knowledge_base = {
    "buffer": "Buffer memory stores all messages verbatim. Simple but grows unbounded and will overflow the context window.",
    "window": "Window memory keeps the last K exchanges. Fixed size but has a sharp cutoff — old context vanishes entirely.",
    "summary": "Summary memory uses an LLM to maintain a running summary. Scales to long chats but costs extra LLM calls.",
    "summary buffer": "Summary buffer memory combines verbatim recent messages with summarized older ones. Best general-purpose default.",
    "entity": "Entity memory tracks specific people, projects, and tools mentioned in conversation. Great for CRM-like apps.",
    "vector": "VectorStore memory uses embeddings to retrieve semantically similar past exchanges. Best for very long conversations.",
    "memory": "LangChain provides 7 memory types: buffer, window, token buffer, summary, summary buffer, entity, and vector store.",
}

def retrieve(query: str) -> str:
    query_lower = query.lower()
    matched = [doc for key, doc in knowledge_base.items() if key in query_lower]
    return " ".join(matched) if matched else "No relevant documents found."


# Reformulation chain — output ONLY the rephrased question
reformulate_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You rephrase follow-up questions into standalone questions. "
         "Given the conversation history, rewrite the question so it makes sense "
         "without any prior context. "
         "If it is already standalone, return it unchanged. "
         "Respond with ONLY the rephrased question, nothing else. No explanation."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Answer chain — answer ONLY from the provided context
answer_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You answer questions using ONLY the provided context below. "
         "Do not use any outside knowledge. "
         "If the context does not contain the answer, say 'Not found in knowledge base.' "
         "Your response must be under 40 words.\n\n"
         "Context: {context}"),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Memory
rag_history = []

def convo_rag(user_question):
    # 1. Reformulate with history
    standalone = reformulate_chain.invoke({
        "chat_history": rag_history,
        "question": user_question
    })

    # 2. Retrieve
    context = retrieve(standalone)

    # 3. Answer
    answer = answer_chain.invoke({"context": context, "question": standalone})

    # 4. Save to memory
    rag_history.append(HumanMessage(content=user_question))
    rag_history.append(AIMessage(content=answer))

    return standalone, answer


# Test with follow-up questions
turns = [
    "What is buffer memory?",
    "What's the main problem with it?",        # "it" = buffer memory
    "What's a better alternative?",              # better than buffer
    "Tell me about the summary buffer type.",
    "Why is it considered the best default?",    # "it" = summary buffer
]

for i, q in enumerate(turns, 1):
    standalone, answer = convo_rag(q)
    
    print(f"\nTurn {i}:")
    print(f"  Original:     {q}")
    if standalone.strip() != q.strip():
        print(f"  Reformulated: {standalone}")
    print(f"  Answer:       {answer}")


Turn 1:
  Original:     What is buffer memory?
  Reformulated: What is buffer memory used for in computing?
  Answer:       Buffer memory is used to store all messages verbatim, allowing for simple storage but growing unbounded and potentially causing overflow.

Turn 2:
  Original:     What's the main problem with it?
  Reformulated: What problems arise when messages are stored verbatim?
  Answer:       Not found in knowledge base.

Turn 3:
  Original:     What's a better alternative?
  Reformulated: Can you use a more efficient data structure, such as a ring buffer?
  Answer:       Not found in knowledge base.

Turn 4:
  Original:     Tell me about the summary buffer type.
  Reformulated: What is a summary buffer?
  Answer:       A summary buffer combines verbatim recent messages with summarized older ones, serving as the best general-purpose default.

Turn 5:
  Original:     Why is it considered the best default?
  Reformulated: What advantages does the summary buffer type offer ove

---

## 9. Choosing the Right Memory Type — Decision Framework

### Step-by-Step Decision

**Step 1 — Conversation Length:**
- 1–15 turns → Buffer Memory
- 15–50 turns → Window or Token Buffer
- 50+ turns → Summary Buffer

**Step 2 — Context Requirements:**
- Need exact recent messages? → Buffer, Window, or Summary Buffer
- Need old context preserved? → Summary, Summary Buffer, or Entity
- Need topic-based recall? → VectorStore

**Step 3 — Cost Constraints:**
- Minimal budget → Buffer or Window (no extra LLM calls)
- Moderate budget → Summary Buffer
- Generous budget → Entity or VectorStore

> **Default recommendation:** Start with **ConversationSummaryBufferMemory**. It handles the widest range of scenarios.

### Experiment 9A: Side-by-Side Comparison

In [12]:
# Compare Buffer vs Window vs Summary Buffer side by side

# Reset all memories
buf = BufferMemory()
win = WindowMemory(k=2)
sb = SummaryBufferMemory(llm, max_recent=4)

memories = [
    ("Buffer", buf),
    ("Window(k=2)", win),
    ("SummaryBuffer", sb),
]

# Simulate a conversation
turns = [
    "My name is Alex.",
    "I love Python programming.",
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "What is my name?",  # Key test — who remembers?
]

for i, msg in enumerate(turns, 1):
    print(f"\n{'=' * 60}")
    print(f"Turn {i}: {msg}")
    print(f"{'=' * 60}")

    for name, mem in memories:
        if isinstance(mem, SummaryBufferMemory):
            data = mem.load()
            response = sb_chain.invoke({
                "summary": data["summary"],
                "recent": data["recent"],
                "input": msg
            })
        else:
            response = convo_chain.invoke({
                "history": mem.load(),
                "input": msg
            })
        mem.save(msg, response)
        print(f"  {name:20s} → {response[:80]}")

print(f"\n{'=' * 60}")
print("MEMORY STATES AFTER 5 TURNS:")
print(f"{'=' * 60}")
print(f"  Buffer:        {buf.size} messages (all stored)")
print(f"  Window(k=2):   sees {win.window_size} of {win.total_stored} messages")
print(f"  SummaryBuffer: {len(sb.load()['recent'])} recent + summary")
if sb.summary:
    print(f"  Summary text:  {sb.summary}...")


Turn 1: My name is Alex.
  Buffer               → Nice to meet you, Alex! How can I assist you today?
  Window(k=2)          → It's nice to meet you, Alex. I'll do my best to assist you with any questions or
  SummaryBuffer        → It's nice to meet you, Alex. Is there something I can help you with today?

Turn 2: I love Python programming.
  Buffer               → Python is a popular and versatile language, making it a great choice for beginne
  Window(k=2)          → Python is a fantastic language to learn, known for its simplicity and versatilit
  SummaryBuffer        → Python is a popular and versatile language, known for its simplicity and extensi

Turn 3: What is 2 + 2?
  Buffer               → The answer is 4. Simple math question, but great for a quick check!
  Window(k=2)          → The answer to 2 + 2 in Python is 4, as it follows the standard arithmetic rules 
  SummaryBuffer        → The answer to 2 + 2 is 4, a simple yet fundamental math problem that's often use

Turn 4:

---

## 10. Persistent Memory & Advanced Patterns

All memory types above are **in-memory** — lost on restart. Production apps need persistence.

### Persistence Backends

| Backend | Use Case |
|---------|----------|
| **Redis** | Fast session-based memory, TTL support |
| **PostgreSQL** | Structured storage, ACID compliance |
| **MongoDB** | Flexible document storage |
| **SQLite** | Local dev/testing |

### Advanced Patterns

| Pattern | Description |
|---------|-------------|
| **Multi-Memory** | Window + Entity + VectorStore simultaneously |
| **Token Budget Mgmt** | Dynamically adjust memory to available context space |
| **Filtered Memory** | Remove PII, skip small talk, deduplicate |
| **Cross-Session** | Persist entity/vector memory across sessions for personalization |

### Memory Best Practices

| Practice | Description |
|----------|-------------|
| Start with Summary Buffer | Best default for most applications |
| Set explicit token budgets | Don't let memory grow unbounded |
| Use cheaper models for memory ops | Summary updates don't need your best model |
| Persist memory in production | In-memory is for development only |
| Reformulate queries in RAG | Follow-ups with pronouns will fail without it |
| Key memory by session ID | Never share memory across unrelated conversations |
| Test with long conversations | Memory bugs surface after many turns (50+) |

### Common Pitfalls

| Pitfall | Solution |
|---------|----------|
| Buffer for long chats | Switch to Summary Buffer |
| No reformulation in conv. RAG | Add question reformulation step |
| Ignoring token budgets | Set explicit `max_token_limit` |
| Same model for summaries | Use a smaller model for memory ops |
| No session isolation | Key memory by unique session ID |
| No persistence | Persist to Redis/PostgreSQL |

---

## 11. Sandbox — Try It Yourself!

In [13]:
# ============================================================
#  SANDBOX - Try different memory types!
# ============================================================

# Choose a memory type
memory_type = "summary_buffer"  # Options: "buffer", "window", "summary_buffer"

# ============================================================

if memory_type == "buffer":
    mem = BufferMemory()
elif memory_type == "window":
    mem = WindowMemory(k=3)
elif memory_type == "summary_buffer":
    mem = SummaryBufferMemory(llm, max_recent=4)

my_question = "Tell me about Python programming."

if isinstance(mem, SummaryBufferMemory):
    data = mem.load()
    response = sb_chain.invoke({
        "summary": data["summary"],
        "recent": data["recent"],
        "input": my_question
    })
else:
    response = convo_chain.invoke({
        "history": mem.load(),
        "input": my_question
    })

mem.save(my_question, response)

print(f"[{memory_type.upper()}]")
print(f"Q: {my_question}")
print(f"A: {response}")

[SUMMARY_BUFFER]
Q: Tell me about Python programming.
A: Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility, making it a popular choice for beginners and experienced developers alike. Its extensive libraries and frameworks, such as NumPy, pandas, and Django, have further cemented its position as a leading language in the industry.


---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LLMs are stateless** | Every API call is isolated — memory bridges this gap |
| **MessagesPlaceholder** | The standard way to inject history into prompts |
| **Buffer Memory** | Stores everything — simple, perfect recall, but overflows |
| **Window Memory** | Last K turns — fixed size, sharp "cliff" cutoff |
| **Token Buffer** | Like window but token-counted — precise budget control |
| **Summary Memory** | LLM-compressed — scales to long chats, costs extra calls |
| **Summary Buffer** | Verbatim recent + summarized old — **best general default** |
| **Entity Memory** | Tracks people/projects/tools — great for CRM-like apps |
| **VectorStore Memory** | Semantic similarity retrieval — best for very long chats |
| **Question Reformulation** | **Mandatory** for conversational RAG — pronouns break retrieval |
| **Token Budgets** | Memory, retrieval, system prompt, response all compete for context |
| **Persistence** | In-memory is dev only — use Redis/PostgreSQL in production |
| **Default Choice** | Start with Summary Buffer, specialize only if needed |